# Experiment 09 — Cross-policy distinctiveness

**Paper reference.** §2.2 of v15 (validating that the six prompts induce distinguishable policies).

**Goal.** Train a linear model on policy i, test on policy j. Diagonal entries should dominate — if not, the prompts induce nearly identical behaviour and the narrative of '6 distinct policies' overstates the reality.

In [ ]:
import os, sys, warnings
warnings.simplefilter('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
import negolin_v15 as nv
nv.setup_matplotlib()

DATA_DIR = '/home/claude/data/merged_results'    # <-- edit if needed
OUT_DIR  = './out'
os.makedirs(OUT_DIR, exist_ok=True)

raw = nv.load_run_files(DATA_DIR)
_, turns = nv.preprocess_turns(raw)
POLICIES = list(nv.POLICIES)
SCHEMES  = list(nv.SCHEMES)
print(f'rows={len(turns):,}  policies={POLICIES}  schemes={SCHEMES}')

In [ ]:
# Load the Stage-2 selection produced by 02_main_sweep.ipynb
sel_path = Path(OUT_DIR) / 'selected_linear.csv'
if sel_path.exists():
    SELECTED = pd.read_csv(sel_path)
    print('Loaded selected_linear.csv:'); display(SELECTED.head())
else:
    # Sensible fallback: Feature II + Ridge per (scheme, policy)
    rows = [{'scheme': s, 'policy': p, 'chosen_family': 'II', 'chosen_model': 'Ridge'}
            for s in SCHEMES for p in POLICIES]
    SELECTED = pd.DataFrame(rows)
    print('selected_linear.csv not found -> using fallback (II/Ridge)')

def picked(scheme, policy):
    r = SELECTED.set_index(['scheme','policy']).loc[(scheme, policy)]
    fam = r['chosen_family']
    if fam == 'Base': fam = 'II'    # Base isn't a feature spec -- fall back
    return fam, r['chosen_model']

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

# Standardise everyone on the same feature spec for a clean comparison
COMMON_FAM = 'II'

def design_for(scheme, policy, fam=COMMON_FAM):
    sub = turns.query('scheme == @scheme and policy == @policy')
    meta, X, y, _ = nv.build_design_matrix(sub, fam)
    (_, _, _), (m_n, X_n, y_n) = nv.partition_persist(meta, X, y)
    return m_n, X_n, y_n

def transfer_matrix_policies(scheme):
    designs = {p: design_for(scheme, p) for p in POLICIES}
    mat = pd.DataFrame(np.nan, index=POLICIES, columns=POLICIES, dtype=float)
    for p_train in POLICIES:
        m_tr, X_tr, y_tr = designs[p_train]
        if len(y_tr) < 30: continue
        est = Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=1.0))]).fit(X_tr, y_tr)
        for p_test in POLICIES:
            m_te, X_te, y_te = designs[p_test]
            if len(y_te) < 5: continue
            mat.loc[p_train, p_test] = float(np.mean(np.abs(est.predict(X_te) - y_te)))
    return mat

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, sch in zip(axes, SCHEMES):
    M = transfer_matrix_policies(sch)
    im = ax.imshow(M.values, cmap='viridis_r', aspect='auto')
    ax.set_title(f'scheme {sch}')
    ax.set_xticks(range(len(POLICIES))); ax.set_xticklabels(POLICIES, rotation=45)
    ax.set_yticks(range(len(POLICIES))); ax.set_yticklabels(POLICIES)
    for i in range(len(POLICIES)):
        for j in range(len(POLICIES)):
            v = M.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7,
                        color='white' if v > np.nanmean(M.values) else 'black')
    fig.colorbar(im, ax=ax, fraction=0.04)
fig.suptitle(f'Train on row policy, test on column policy (family {COMMON_FAM} + Ridge, Y-scale MAE)', fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig_exp09_policy_transfer.png'))
plt.show()

If the diagonal is consistently lower than its row and column maxima, the linear policies have **distinguishable signatures**. Specifically, the diagonal advantage is what lets you say in §3.4 that a policy's identity can be recovered from a moderate sample of its negotiation traces.